If you are building a web backend (which we will do in Module 3), a synchronous wait means no other user can access your app until the first user's LLM response is fully generated. **Asynchronous programming solves this. It allows Python to say, "Hey, I'm waiting on the network for this LLM. While I wait, I'm going to go do other tasks."**

In [1]:
# Synchronus Solution

import time

def fake_llm_call(task_id):
    print(f"Task {task_id}: Starting...")
    time.sleep(2) # Simulating network wait synchronously (blocking)
    print(f"Task {task_id}: Finished!")

start_time = time.time()

# Running them one by one
fake_llm_call(1)
fake_llm_call(2)
fake_llm_call(3)

print(f"Total Synchronous Time: {time.time() - start_time:.2f} seconds")

Task 1: Starting...
Task 1: Finished!
Task 2: Starting...
Task 2: Finished!
Task 3: Starting...
Task 3: Finished!
Total Synchronous Time: 6.02 seconds


In [ ]:
import asyncio
import time

# We use async def to define an asynchronous function

async def async_fake_llm_call(task_id):
    print(f"Task {task_id}: Starting...")
    await asyncio.sleep(2) # Simulating network wait asynchronously (non-blocking)
    print(f"Task {task_id}: Finished!")

start_time = time.time()

# asyncio.gather runs multiple async tasks concurrently
# use await whenever we have an operation that takes time (like network I/O or a sleep)
await asyncio.gather(
    async_fake_llm_call(1),
    async_fake_llm_call(2),
    async_fake_llm_call(3)
)

print(f"Total Asynchronous Time: {time.time() - start_time:.2f} seconds")

Task 1: Starting...
Task 2: Starting...
Task 3: Starting...
Task 1: Finished!
Task 2: Finished!
Task 3: Finished!
Total Asynchronous Time: 2.00 seconds


In [ ]:
# full example

import httpx
import asyncio
import time

from sympy import Line

# async def: This tells Python, "This function contains operations that will take time, so be prepared to pause it and do other things." 
    # Functions defined this way are called coroutines.
# await client.get(url): 
    # This is the most crucial line. When Python hits await, it essentially says: "I just sent a request to the internet. 
    # It's going to take a few hundred milliseconds to come back. Instead of freezing the whole program, 
    # I am going to pause just this specific task and go see if there is other work I can do."
async def fetch_data(client, url, task_id):
    print(f"Task {task_id}: Requesting data...")
    response = await client.get(url)
    print(f"Task {task_id}: Received status {response.status_code}")

async def main():
    urls = [
        "https://jsonplaceholder.typicode.com/posts/1",
        "https://jsonplaceholder.typicode.com/posts/2",
        "https://jsonplaceholder.typicode.com/posts/3"
    ]
    
    start_time = time.time()
    
    # If we just used a regular with httpx.AsyncClient() as client:, 
        # Python would completely freeze our program while it waited for that initial connection to open, and freeze it again when closing it.
    # When Python hits async with (or any await), it pauses the entire current function and hands control back to the Event Loop (the master orchestrator).
    # Example:
        # Task A: Bake a pizza.
        # Task B: Make a salad.
        # You start Task A. The recipe says:
            # Line 1: Knead dough.
            # Line 2: Put in the oven and wait 10 minutes (await oven_timer()).
            # Line 3: Take out and slice.
        # When you hit Line 2, you put the pizza in the oven. You do not immediately skip to Line 3 and try to slice an unbaked pizza!
            # Instead, you pause Task A entirely. You step back, look at your ticket rail (the Event Loop), see Task B, and go make the salad. 
            # When the oven dings 10 minutes later, you stop what you're doing, resume Task A, and finally execute Line 3 to slice the pizza.


    async with httpx.AsyncClient() as client:
        # Create a list of tasks to run concurrently
        tasks = [fetch_data(client, urls[i], i+1) for i in range(3)]
        # asyncio.gather takes our list of pending jobs, tosses them all into Python's "Event Loop," and says, "Start all of these right now, concurrently."
        await asyncio.gather(*tasks)
        
    print(f"Total Network Time: {time.time() - start_time:.2f} seconds")

# In a standard python script you'd use asyncio.run(main()), 
# but in Jupyter we can just await it directly!
await main()

Task 1: Requesting data...
Task 2: Requesting data...
Task 3: Requesting data...
Task 2: Received status 200
Task 3: Received status 200
Task 1: Received status 200
Total Network Time: 0.17 seconds
